# 网络搜索

> 没有一条帆船能吹动自己行驶，它需要外面的风。 -- 因可觅[《量子离歌》](https://read.douban.com/ebook/12945743/)

网络搜索本身是一个很重的服务，它包含三项核心任务：网页爬取、网页索引和网页检索。**网页索引** 几乎只能由大公司完成。即使 [SearXNG](https://github.com/searxng/searxng) 这样的开源检索方案，也只是聚合第三方搜索引擎（如必应、搜狗）的结果，没有独立索引能力。因此，即使独立开发网络搜索功能，也难以绕开大公司的服务。不如一开始就老老实实用大公司的 API，来获取搜索结果。

本节介绍三种网络搜索方式：

- 使用 [DashScope](https://dashscope.aliyun.com/)
- 使用 [Tavily](https://www.tavily.com/)
- 使用 [DDGS](https://github.com/deedy5/ddgs)

**DashScope** 是阿里的 MaaS 平台，由于我们已经有了阿里的 `API_KEY`，这对我们是最轻松的方式。**Tavily** 是 LangChain [推荐](https://docs.langchain.com/oss/python/integrations/retrievers/tavily) 的搜索服务，有免费额度，但需要花时间申请 `API_KEY`。**DDGS** 是一个免费的搜索引擎，不需要 `API_KEY`，但搜索结果的质量较差。

下面来看这三种搜索服务的使用方法和效果。

In [ ]:
import os
from langchain.agents import create_agent
from langchain.tools import tool

# Load shared project config from the repository-root .env
from pathlib import Path
import sys


def find_dive_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "dive_config.py").exists():
            return candidate
        nested = candidate / "LangChain" / "projects" / "dive-into-langgraph"
        if (nested / "dive_config.py").exists():
            return nested
    raise RuntimeError(
        "Start this notebook from the repository root or the "
        "dive-into-langgraph project directory."
    )


PROJECT_ROOT = find_dive_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dive_config import create_chat_model, create_embeddings, get_model_config, load_project_env

load_project_env(PROJECT_ROOT)
model_config = get_model_config()

# 加载模型
llm = create_chat_model(
    model=os.getenv("DIVE_WEB_SEARCH_CHAT_MODEL") or model_config.model,
    temperature=0.7,
)


In [ ]:
# !pip install dashscope langchain-community tavily-python ddgs


## 一、使用 DashScope

首先安装 dashscope：

```bash
pip install dashscope
```

只需要将 `Generation.call` 函数的网络参数 `enable_search` 设为 `True` 就能接入互联网。下面把这个函数做成工具供 Agent 调用。

In [ ]:
from dashscope import Generation

@tool
def dashscope_search(query: str) -> str:
    """
    使用夸克搜索 API 搜索互联网信息。
    """
    response = Generation.call(
        model='qwen3-max',
        prompt=query,
        enable_search=True,
        result_format='message'
    )

    if response.status_code == 200:
        return response.output.choices[0].message.content
    else:
        return (
            "Search failed with status code: "
            f"{response.status_code}, message: {response.message}"
        )

# 创建 Agent
agent = create_agent(
    model=llm,
    tools=[dashscope_search],
    system_prompt="你是一个智能助手，回答前必须使用工具搜索互联网信息",
)

# 运行 Agent
response = agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "告诉我今天的日期，以及今天最重要的一条新闻"
    }]}
)


In [ ]:
# 获取 Agent 的全部回复
for message in response['messages']:
    message.pretty_print()


## 二、使用 Tavily

首先安装 tavily：

```bash
pip install langchain-community tavily-python
```

然后从 [Tavily](https://www.tavily.com/) 官网申请 `API_KEY`，填入下面的方框中。

In [ ]:
import getpass
os.environ["TAVILY_API_KEY"] = getpass.getpass()


In [ ]:
from langchain_community.retrievers import TavilySearchAPIRetriever

retriever = TavilySearchAPIRetriever(k=3)
query = "告诉我今天的日期，以及今天最重要的一条新闻"


返回的内容有点多，可以解开下面这行命令的注释查看👇

In [ ]:
# retriever.invoke(query)


同样把 Tavily 作为工具加入 Agent 看看效果怎么样！

In [ ]:
prompt_template = """基于下面的搜索结果回答问题：

搜索结果：{context}

问题：{question}"""

def format_docs(docs):
    """合并 Tavily 的搜索结果"""
    return "\n\n".join(doc.page_content for doc in docs)

@tool
def tavily_search(query: str) -> str:
    """
    使用 Tavily 搜索 API 搜索互联网信息。
    """
    retriever = TavilySearchAPIRetriever(k=3)
    contents = retriever.invoke(query)
    if len(contents) > 0:
        return format_docs(contents)
    else:
        return "没有搜索结果"

# 创建 Agent
agent = create_agent(
    model=llm,
    tools=[tavily_search],
    system_prompt="你是一个智能助手，回答前必须使用工具搜索互联网信息",
)

# 运行 Agent
response = agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "告诉我今天的日期，以及今天最重要的一条新闻"
    }]}
)


In [ ]:
# 获取 Agent 的全部回复
for message in response['messages']:
    message.pretty_print()


## 三、使用 DDGS

首先安装 DDGS：

```bash
pip install ddgs
```

我们直接使用 DDGS 获取的摘要信息，没有做进一步网页爬取。下面将 DDGS 做成工具。

In [ ]:
from ddgs import DDGS

# 创建 ddgs 客户端
ddgs = DDGS()

@tool
def ddgs_search(query: str) -> str:
    """
    使用 DDGS 搜索 API 搜索互联网信息。

    Args:
        query: 搜索关键词或问题。
        max_results: 返回的最大结果条数。

    Returns:
        包含每条搜索结果的标题、摘要与链接的字符串。
    """
    results = list(
        ddgs.text(
            query=query,
            region="wt-wt",  # wt-wt zh
            timelimit='y',
            safesearch='off',  # moderate off
            page=1,
            backend='auto',
            max_results=3,
        )
    )

    content = ""
    for i, r in enumerate(results, 1):
        content += f"【结果 {i}】\n"
        content += f"标题: {r['title']}\n"
        content += f"摘要: {r['body']}\n"
        content += f"链接: {r['href']}\n\n"

    return content

# 创建 Agent
agent = create_agent(
    model=llm,
    tools=[ddgs_search],
    system_prompt="你是一个智能助手，回答前必须使用工具搜索互联网信息",
)

# 运行 Agent
response = agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "告诉我今天的日期，以及今天最重要的一条新闻"
    }]}
)


In [ ]:
# 获取 Agent 的全部回复
for message in response['messages']:
    message.pretty_print()
